<a href="https://colab.research.google.com/github/srinath698/zepto-analytics-pipeline/blob/main/zepto_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df=pd.read_csv('/content/zepto_v2.csv')

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('zepto_v2.csv')
df.columns = ['category', 'name', 'mrp', 'discount_pct', 'available_qty',
              'discounted_selling_price', 'weight_gms', 'out_of_stock', 'quantity']

print("="*70)
print("STEP 1: INITIAL DATA OVERVIEW")
print("="*70)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nData types:\n{df.dtypes}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nDuplicate SKU names: {df.duplicated(subset='name').sum()}")


invalid_rows = (df['mrp'] == 0).sum()
df = df[df['mrp'] != 0].copy()
print(f"\nRemoved {invalid_rows} rows with MRP = 0")

df['mrp'] = df['mrp'] / 100.0
df['discounted_selling_price'] = df['discounted_selling_price'] / 100.0

df['discount_amount'] = df['mrp'] - df['discounted_selling_price']
df['revenue_potential'] = df['discounted_selling_price'] * df['available_qty']

conditions = [
    df['weight_gms'] < 1000,
    df['weight_gms'] < 5000
]
choices = ['Low', 'Medium']
df['weight_category'] = np.select(conditions, choices, default='Bulk')

df['price_tier'] = np.where(df['mrp'] >= 500, 'Premium',
                     np.where(df['mrp'] >= 200, 'Mid', 'Budget'))

print("\n" + "="*70)
print("STEP 2: FEATURE ENGINEERING COMPLETE")
print("="*70)
print(df[['name','mrp','discount_amount','revenue_potential','weight_category','price_tier']].head())
print("\n" + "="*70)
print("STEP 3: DESCRIPTIVE STATISTICS")
print("="*70)

numeric_cols = ['mrp', 'discount_pct', 'discounted_selling_price', 'weight_gms']
stats_summary = pd.DataFrame({
    'mean':   df[numeric_cols].apply(np.mean),
    'median': df[numeric_cols].apply(np.median),
    'std':    df[numeric_cols].apply(np.std),
    'min':    df[numeric_cols].apply(np.min),
    'max':    df[numeric_cols].apply(np.max),
    'p25':    df[numeric_cols].apply(lambda x: np.percentile(x, 25)),
    'p75':    df[numeric_cols].apply(lambda x: np.percentile(x, 75)),
    'p95':    df[numeric_cols].apply(lambda x: np.percentile(x, 95)),
})
print(stats_summary)


print("\n" + "="*70)
print("STEP 4: OUTLIER DETECTION (MRP) - IQR METHOD")
print("="*70)

q1, q3 = np.percentile(df['mrp'], [25, 75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = df[(df['mrp'] < lower_bound) | (df['mrp'] > upper_bound)]
print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of MRP outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}% of data)")
print(outliers.nlargest(5, 'mrp')[['name','category','mrp']])


print("\n" + "="*70)
print("STEP 5: CATEGORY-LEVEL INSIGHTS")
print("="*70)

category_summary = df.groupby('category').agg(
    sku_count=('name', 'count'),
    avg_mrp=('mrp', 'mean'),
    avg_discount_pct=('discount_pct', 'mean'),
    total_revenue_potential=('revenue_potential', 'sum'),
    out_of_stock_rate=('out_of_stock', 'mean'),
    total_weight_kg=('weight_gms', lambda x: np.sum(x)/1000)
).round(2).sort_values('total_revenue_potential', ascending=False)

category_summary['out_of_stock_rate'] = (category_summary['out_of_stock_rate']*100).round(2)
print(category_summary)

print("\n" + "="*70)
print("STEP 6: CORRELATION MATRIX")
print("="*70)
corr_matrix = np.corrcoef(df[['mrp','discount_pct','discounted_selling_price','weight_gms']].T)
corr_df = pd.DataFrame(corr_matrix,
                        index=['mrp','discount_pct','disc_price','weight'],
                        columns=['mrp','discount_pct','disc_price','weight'])
print(corr_df.round(3))


print("\n" + "="*70)
print("STEP 7: AUTO-GENERATED KEY INSIGHTS")
print("="*70)

top_revenue_cat = category_summary.index[0]
top_revenue_val = category_summary.iloc[0]['total_revenue_potential']

high_mrp_oos = df[(df['mrp'] > 300) & (df['out_of_stock'] == True)]
high_mrp_oos_lost_value = (high_mrp_oos['mrp'] * 1).sum()

margin_heavy = df[(df['mrp'] > 500) & (df['discount_pct'] < 10)]

highest_oos_cat = category_summary['out_of_stock_rate'].idxmax()
highest_oos_val = category_summary['out_of_stock_rate'].max()

heaviest_cat = category_summary['total_weight_kg'].idxmax()
heaviest_val = category_summary['total_weight_kg'].max()
total_weight_all = category_summary['total_weight_kg'].sum()

insights = f"""
1. REVENUE: '{top_revenue_cat}' is the top revenue-potential category at ₹{top_revenue_val:,.0f}.

2. STOCKOUTS: {len(high_mrp_oos)} high-value products (MRP > ₹300) are currently
   out of stock — representing ₹{high_mrp_oos_lost_value:,.0f} in listed MRP value
   sitting unavailable to customers.

3. MARGIN OPPORTUNITY: {len(margin_heavy)} products priced above ₹500 carry less
   than 10% discount — these are high-margin SKUs worth protecting/promoting.

4. STOCK RISK: '{highest_oos_cat}' has the highest out-of-stock rate at
   {highest_oos_val:.1f}% of its SKUs — a potential fulfillment/supply issue.

5. LOGISTICS LOAD: '{heaviest_cat}' accounts for {heaviest_val:,.0f} kg
   ({heaviest_val/total_weight_all*100:.1f}% of total inventory weight) —
   relevant for warehouse and last-mile delivery planning.

6. PRICE-DISCOUNT RELATIONSHIP: correlation between MRP and discount % is
   {corr_df.loc['mrp','discount_pct']:.2f}, suggesting {"higher-priced items tend to get discounted more" if corr_df.loc['mrp','discount_pct']>0.2 else "little to no strong linear relationship between price and discount level"}.
"""
print(insights)


df.to_csv('zepto_cleaned.csv', index=False)
print("Saved enriched dataset -> zepto_cleaned.csv")

STEP 1: INITIAL DATA OVERVIEW
Shape: 3732 rows x 9 columns

Data types:
category                    object
name                        object
mrp                          int64
discount_pct                 int64
available_qty                int64
discounted_selling_price     int64
weight_gms                   int64
out_of_stock                  bool
quantity                     int64
dtype: object

Null values:
category                    0
name                        0
mrp                         0
discount_pct                0
available_qty               0
discounted_selling_price    0
weight_gms                  0
out_of_stock                0
quantity                    0
dtype: int64

Duplicate SKU names: 2051

Removed 1 rows with MRP = 0

STEP 2: FEATURE ENGINEERING COMPLETE
               name   mrp  discount_amount  revenue_potential weight_category  \
0             Onion 25.00             4.00              63.00          Medium   
1     Tomato Hybrid 42.00             7.00    